In [2]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(1)

producer.flush()
producer.close()

Writing producer.py


In [1]:
%%file consumer_speed.py
from kafka import KafkaConsumer
from collections import defaultdict, deque
from datetime import datetime
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='consumer-speed-v1',
    auto_offset_reset='latest',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

user_events = defaultdict(deque)

print("Nasłuchuję anomalii prędkości...")

for message in consumer:
    tx = message.value
    user_id = tx['user_id']
    ts = datetime.fromisoformat(tx['timestamp'])

    user_events[user_id].append(ts)

    while user_events[user_id] and (ts - user_events[user_id][0]).total_seconds() > 60:
        user_events[user_id].popleft()

    if len(user_events[user_id]) > 3:
        print(f"ALERT: {user_id} wykonał {len(user_events[user_id])} transakcje w ciągu 60 sekund")

Writing consumer_speed.py
